# 配电终端通信异常诊断模型的分层划分、标准化与 F1 测试 · 教学参考

这是帮助理解写法的参考实现，不是官方标准答案。请在 `answer.ipynb` 独立作答；不要在本文件中覆盖自己的草稿。

## 逐步理解

1. read_csv 要知道从哪里读取数据。题目已给出文件路径；代码外面已有引号，所以空里不用再加引号。

2. dropna 会删除含缺失值的行。题目已经写好后面的括号，这里只需填方法名；接下来还会去重和重新编号。

3. 心跳间隔、丢失率等 9 列是判断通信异常的线索，合称特征 X。多列要写成带方括号的列表。实际 CSV 列名是“CPU利用率”和“内存利用率”，中间都没有空格。

4. “是否异常”是模型要预测的正确答案列，称为标签 y。列名是文字，这一空要带引号。

5. test_size 是留给测试的数据比例。0.2 表示 20% 测试、80% 训练，对应题目要求的 8:2。

6. stratify=y 表示按通信异常与正常的比例分层抽样，让训练集和测试集中的异常样本比例尽量接近。

7. 训练集要先让标准化器学习规则（fit），再按规则转换数据（transform），合起来就是 fit_transform。

8. 测试集只使用训练集学到的标准化规则进行转换，不能再次 fit；否则就提前用测试数据参与学习了。

9. fit 是训练：把终端通信数据 X_train_s 和已知的异常结果 y_train 交给逻辑回归模型学习。

10. predict 是预测：让训练好的模型根据测试集线索 X_test_s 给出判断，结果存在 y_pred。

11. accuracy_score 将预测和真实答案比较，算出总体有多少比例判断对了，这叫准确率。

12. f1_score 同时考虑异常判断是否准确，以及真正的通信异常有没有被找到；不能只看总体准确率。

13. 0.80 是这道通信异常题指定的 F1 合格线。程序算出的 F1 达到它才显示“达到要求”。

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression

data = pd.read_csv("04/SS-5-4-4-04_terminal_comm.csv")
data = data.dropna().drop_duplicates().reset_index(drop=True)
X = data[["心跳间隔", "心跳丢失率", "信号强度", "误码率", "重传次数", "响应延迟", "CPU利用率", "内存利用率", "在线时长"]]
y = data["是否异常"]
X_train, X_test, y_train, y_test = train_test_split(X, y,
test_size=0.2, random_state=2026, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
model = LogisticRegression(max_iter=2000, random_state=2026)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print("准确率：", accuracy)
print("F1：", f1)
print("F1 达到要求" if f1 >= 0.80 else "F1 未达到要求")
